# B2-flip — donor thống nhất, sửa checkpoint rồi chạy tiếp

Gắn **bundle mới** `grace_b2_flip_frozen_donor_inputs.zip`, **checkpoint I-06** `grace_b2_flip_checkpoint.zip` và model Qwen GGUF Q4_K_M. Notebook tự tìm Input, không phụ thuộc username.

Mặc định resume: giữ 5.202 output còn khớp, chạy lại **28 Devign validation + 5 Devign test** đã đổi prompt trước, rồi tiếp tục **4.776 mẫu thiếu** bằng prompt mới. Sau 28 validation, ghép đủ 2.732 để khóa ngưỡng mới trước test. Tổng bổ sung cho checkpoint I-06: **4.809 lượt LLM**. Dừng/tiếp tục nhiều phiên vẫn giữ hàng đợi ưu tiên và không gọi lại mẫu mới đã hoàn thành.

Ngân sách 9 giờ 35 phút, chừa 5 phút đóng gói; một mẫu đang chạy có thể vượt thời điểm dừng. Checkpoint gốc được lưu trong `history/`; kết quả phiên mới nằm ở thư mục riêng. Bundle cũ trong Input được bỏ qua bằng kiểm tra phiên donor.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, traceback, shutil, platform, importlib.metadata, zipfile, hashlib, time, re
from datetime import datetime, timezone

DIAG = Path('/kaggle/working/grace_diagnostics')
DIAG.mkdir(parents=True, exist_ok=True)
EVENTS = DIAG / 'events.log'

def note(message):
    text = str(message)
    print(text, flush=True)
    with EVENTS.open('a', encoding='utf-8') as f:
        f.write(f'[{datetime.now(timezone.utc).isoformat()}] {text}\n')

def diagnostics_zip():
    archive = shutil.make_archive('/kaggle/working/grace_diagnostics', 'zip', root_dir=DIAG)
    print('Tải log chẩn đoán từ Kaggle Output:', archive, flush=True)
    return archive

def run_logged(cmd, name, env=None):
    log = DIAG / f'{name}.log'
    note(f'Bắt đầu {name}: {cmd!r} | log đầy đủ: {log}')
    is_inference = name == 'inference_batch'
    with log.open('w', encoding='utf-8') as f:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding='utf-8', errors='replace', bufsize=1, env=env)
        for line in proc.stdout:
            f.write(line); f.flush()
            if re.search(r'\b(?:Found|Skip(?:ping)?) link\b', line, flags=re.IGNORECASE):
                continue
            if is_inference:
                match = re.search(r'^(\S+).*\((\d+)/(\d+)\)\s*$', line.strip())
                if match:
                    run_key, done, total = match.group(1), int(match.group(2)), int(match.group(3))
                    if done % 100 == 0 or done == total:
                        print(f'Inference {run_key}: {done}/{total}', flush=True)
                elif line.strip():
                    print(line, end='', flush=True)
            else:
                print(line, end='', flush=True)
        code = proc.wait()
    if is_inference and '--status' in cmd:
        status_path = Path(cmd[cmd.index('--status') + 1])
        if status_path.is_file():
            status = json.loads(status_path.read_text(encoding='utf-8'))
            print(f"Inference summary: {status.get('completed_records_total', 0)} records; state={status.get('state')}", flush=True)
            for run_key, progress in status.get('runs', {}).items():
                print(f"  {run_key}: {progress.get('completed', 0)}/{progress.get('count', '?')} [{progress.get('state', 'pending')}]")
    note(f'{name} kết thúc với mã {code}')
    if code:
        raise subprocess.CalledProcessError(code, cmd)

def versions():
    result = {}
    for name in ('llama-cpp-python', 'transformers', 'huggingface_hub', 'scikit-learn', 'numpy', 'scipy'):
        try: result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: result[name] = 'not installed'
    note('Installed package versions: ' + json.dumps(result))
    return result

def capture_failure(shell, etype, evalue, tb, tb_offset=None):
    trace = ''.join(traceback.format_exception(etype, evalue, tb))
    try:
        with (DIAG / 'exceptions.log').open('a', encoding='utf-8') as f:
            f.write(f'\n[{datetime.now(timezone.utc).isoformat()}]\n{trace}\n')
        note(f'Lỗi: {etype.__name__}: {evalue}')
        versions()
        diagnostics_zip()
    except Exception as diagnostic_error:
        print('Không thể ghi đầy đủ chẩn đoán:', diagnostic_error, flush=True)
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    return []

get_ipython().set_custom_exc((Exception,), capture_failure)
note('UTC: ' + datetime.now(timezone.utc).isoformat())
note('Python: ' + sys.version)
note('Executable: ' + sys.executable)
note('Platform: ' + platform.platform())
for loc in ('/kaggle/working', '/kaggle/input'):
    try:
        usage = shutil.disk_usage(loc)
        note(f'Disk {loc}: total={usage.total} free={usage.free} bytes')
    except OSError as exc: note(f'Disk {loc}: {exc}')
versions()
for name, cmd in [('gpu', ['nvidia-smi']), ('cuda_compiler', ['nvcc', '--version']), ('cmake', ['cmake', '--version']), ('pip', [sys.executable, '-m', 'pip', '--version'])]:
    try: run_logged(cmd, name)
    except (OSError, subprocess.CalledProcessError) as exc: note(f'{name}: {type(exc).__name__}: {exc}')
diagnostics_zip()


## Chọn bundle và cấu hình chạy


In [ ]:
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
BUNDLE_WORK = Path('/kaggle/working/grace_b2_flip_frozen_donor_bundles')
MODEL_FILENAME = 'Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf'
RESUME_FROM_CHECKPOINT = True
CHECKPOINT_INPUT_PATH = None  # Chỉ cần đặt nếu gắn nhiều checkpoint cùng tên.
EXPECTED_DONOR_SHA = 'e1d3c6b17c3a0b5c1e05203794d48a33825e0ea00a55deaaef565af25efe0448'
EXPECTED_VARIANT = 'b2_flip_frozen_donor_v1'
if RESUME_FROM_CHECKPOINT:
    if CHECKPOINT_INPUT_PATH is None:
        candidates = sorted(KAGGLE_INPUT_ROOT.rglob('grace_b2_flip_checkpoint.zip'))
        if len(candidates) != 1:
            raise ValueError(f'Gắn đúng một checkpoint B2-flip hoặc đặt CHECKPOINT_INPUT_PATH: {candidates}')
        CHECKPOINT_INPUT_PATH = candidates[0]
    CHECKPOINT_INPUT_PATH = Path(CHECKPOINT_INPUT_PATH)
    if not CHECKPOINT_INPUT_PATH.is_file():
        raise FileNotFoundError(CHECKPOINT_INPUT_PATH)
MAX_RUNTIME_HOURS = 9 + 35/60
CHECKPOINT_RESERVE_SECONDS = 300
GPU_LAYERS = -1
OUT_ROOT = Path('/kaggle/working/grace_b2_flip_frozen_donor_20261101_9h35')
EXPECTED_RUNS = [('devign','validation'), ('reveal','validation'), ('devign','test'), ('reveal','test')]
note(f'Output phiên mới: {OUT_ROOT}; resume={RESUME_FROM_CHECKPOINT}; checkpoint={CHECKPOINT_INPUT_PATH}')


In [ ]:
# Tìm theo tên file và manifest trong Input, không phụ thuộc username hoặc slug Dataset.
BUNDLE_WORK.mkdir(parents=True, exist_ok=True)
found = {}
manifest_paths = sorted(KAGGLE_INPUT_ROOT.rglob('bundle_manifest.json'))
note(f'Tìm thấy {len(manifest_paths)} manifest trong Input; kiểm tra các bundle B2-flip')

def register_bundle(manifest_bytes, prompt_bytes, origin):
    manifest = json.loads(manifest_bytes)
    if (manifest.get('arm') != 'B2-flip' or manifest.get('prompt_variant') != EXPECTED_VARIANT):
        return
    if manifest.get('donor_snapshot_sha256') != EXPECTED_DONOR_SHA:
        raise ValueError(f'Wrong frozen donor snapshot: {origin}')
    key = (manifest.get('dataset'), manifest.get('split'))
    if key not in EXPECTED_RUNS:
        return
    digest = hashlib.sha256(prompt_bytes).hexdigest()
    if digest != manifest.get('prompts_sha256'):
        raise ValueError(f'Prompt hash không khớp manifest: {origin}')
    rows = [json.loads(line) for line in prompt_bytes.decode('utf-8').splitlines() if line.strip()]
    ids = [str(row['id']) for row in rows]
    if len(rows) != manifest.get('count') or len(set(ids)) != len(ids):
        raise ValueError(f'Count/ID không hợp lệ: {origin}')
    if any((row.get('dataset'), row.get('split')) != key for row in rows):
        raise ValueError(f'Prompt row sai dataset/split: {origin}')
    if key in found:
        if found[key]['prompts_sha256'] != digest:
            raise ValueError(f'Trùng bundle khác nội dung cho {key}: {found[key]["origin"]} và {origin}')
        return
    found[key] = {'manifest': manifest_bytes, 'prompts': prompt_bytes,
                  'prompts_sha256': digest, 'count': len(rows), 'origin': str(origin)}

for manifest_path in manifest_paths:
    prompts_path = manifest_path.with_name('prompts.jsonl')
    if prompts_path.is_file():
        register_bundle(manifest_path.read_bytes(), prompts_path.read_bytes(), manifest_path.parent)

# ZIP tổng do notebook chuẩn bị tạo chứa bốn ZIP split; cũng nhận ZIP split riêng.
import io
archive_paths = sorted(set(KAGGLE_INPUT_ROOT.rglob('grace_b2_flip*inputs.zip')) |
                       set(KAGGLE_INPUT_ROOT.rglob('*_b2_flip_kaggle.zip')))
for archive_path in archive_paths:
    note(f'Đọc bundle archive: {archive_path}')
    with zipfile.ZipFile(archive_path) as archive:
        names = set(archive.namelist())
        if {'bundle_manifest.json', 'prompts.jsonl'} <= names:
            register_bundle(archive.read('bundle_manifest.json'), archive.read('prompts.jsonl'), archive_path)
        for member in sorted(name for name in names if name.endswith('_b2_flip_kaggle.zip')):
            with zipfile.ZipFile(io.BytesIO(archive.read(member))) as inner:
                register_bundle(inner.read('bundle_manifest.json'), inner.read('prompts.jsonl'),
                                f'{archive_path}!{member}')

missing = [key for key in EXPECTED_RUNS if key not in found]
if missing:
    raise FileNotFoundError(f'Thiếu bundle {missing} trong Kaggle Input. Add Dataset chứa '
                            'grace_b2_flip_frozen_donor_inputs.zip mới từ notebook 01.')

BUNDLE_PATHS = {}
for dataset, split in EXPECTED_RUNS:
    key = (dataset, split)
    item = found[key]
    dest = BUNDLE_WORK / f'{dataset}_{split}_b2_flip.zip'
    with zipfile.ZipFile(dest, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        z.writestr('bundle_manifest.json', item['manifest'])
        z.writestr('prompts.jsonl', item['prompts'])
    BUNDLE_PATHS[key] = dest
    manifest = json.loads(item['manifest'])
    note(f'Bundle {dataset}/{split}: {item["count"]} rows; missing_graphs={manifest.get("missing_graphs")}; sha256={item["prompts_sha256"]}; source={item["origin"]}')
note(f'Đã xác minh đủ {len(BUNDLE_PATHS)} bundle; OUT_ROOT={OUT_ROOT}')
diagnostics_zip()


## Cài thư viện và xác minh CUDA


In [ ]:
# Cài từng nhóm để nhận diện chính xác bước thất bại; giữ nguyên phiên bản baseline.
# T4 dùng SM75; Kaggle đặt libcuda.so trong /usr/local/nvidia/lib64 nên thêm thư mục này cho CMake.
os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_LIBRARY_PATH=/usr/local/nvidia/lib64'
os.environ['CMAKE_BUILD_PARALLEL_LEVEL'] = '2'
os.environ['FORCE_CMAKE'] = '1'
note('Build config: CMAKE_ARGS=' + os.environ['CMAKE_ARGS'] + '; CMAKE_BUILD_PARALLEL_LEVEL=' + os.environ['CMAKE_BUILD_PARALLEL_LEVEL'] + '; FORCE_CMAKE=' + os.environ['FORCE_CMAKE'])
run_logged([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'transformers==4.57.1', 'huggingface_hub', 'scikit-learn==1.5.2'], 'install_dependencies')
run_logged([sys.executable, '-m', 'pip', 'install', '-v', '--no-cache-dir', '--no-binary', 'llama-cpp-python', 'llama-cpp-python==0.3.16'], 'install_llama_cuda')
versions()
diagnostics_zip()


In [ ]:
import llama_cpp
versions()
note('llama-cpp-python GPU offload: ' + str(llama_cpp.llama_supports_gpu_offload()))
assert llama_cpp.llama_supports_gpu_offload(), 'llama-cpp-python chưa có CUDA; xem install_llama_cuda.log'
diagnostics_zip()


## Tải model và chạy inference


In [ ]:
# Tìm đúng tên file model trong toàn bộ Input đã gắn, bất kể username/slug.
model_files = sorted(path for path in KAGGLE_INPUT_ROOT.rglob(MODEL_FILENAME) if path.is_file())
if len(model_files) != 1:
    raise FileNotFoundError(f'Cần đúng một {MODEL_FILENAME} trong Kaggle Input; '
                            f'tìm thấy: {list(map(str, model_files))}. '
                            'Hãy Add Input chứa model và bỏ bản trùng nếu có; '
                            'notebook không tự tải lại GGUF từ Hugging Face.')
model_path = str(model_files[0])
revision = 'local-input'
note(f'Dùng model input: {model_path}')
print('Model input:', model_path)
print('Batch output directory:', OUT_ROOT)


In [ ]:
%%writefile /kaggle/working/validation_lock.py
"""CPU-only, validation-only threshold locking. Portable into Kaggle notebooks."""
import hashlib
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path

RULES={'version':'validation_mcc_grid_v1','grid':[i/100 for i in range(1,100)],
       'objective':'MCC on finite valid validation scores only','comparison':'score >= threshold',
       'tie_break':'within 1e-9: closest to 0.5, then lower threshold',
       'degenerate':'if every MCC is undefined, lock 0.5 with MCC=null; report explicitly',
       'coverage':'require every validation ID; invalid rows retained in coverage but excluded from fitting',
       'test_usage':'never read test records or metrics to choose threshold'}
def digest_json(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
RULES_SHA256=digest_json(RULES)

def file_sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    with tmp.open('w',encoding='utf-8',newline='\n') as f:
        json.dump(value,f,ensure_ascii=False,indent=2,allow_nan=False);f.write('\n');f.flush();os.fsync(f.fileno())
    os.replace(tmp,path)

def mcc(y,p):
    tp=sum(a==1 and b==1 for a,b in zip(y,p));tn=sum(a==0 and b==0 for a,b in zip(y,p))
    fp=sum(a==0 and b==1 for a,b in zip(y,p));fn=sum(a==1 and b==0 for a,b in zip(y,p))
    den=math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    return (tp*tn-fp*fn)/den if den else None

def pick_threshold(rows):
    valid=[]
    for r in rows:
        value=r.get('s')
        if r.get('valid') and isinstance(value,(int,float)) and math.isfinite(value):
            if not 0<=value<=1:raise ValueError('Score outside [0,1]')
            valid.append(r)
    if not valid:raise ValueError('No finite valid validation scores; cannot lock threshold')
    if {r['label'] for r in valid}!={0,1}:raise ValueError('Validation fitting needs both classes among valid scores')
    y=[r['label'] for r in valid];scores=[r['s'] for r in valid]
    best_t=.5;best_m=None;grid=[]
    for t in RULES['grid']:
        value=mcc(y,[int(v>=t) for v in scores]);grid.append({'threshold':t,'mcc':value})
        if value is None:continue
        if best_m is None or value>best_m+1e-9 or (abs(value-best_m)<=1e-9 and (abs(t-.5),t)<(abs(best_t-.5),best_t)):
            best_t,best_m=t,value
    return {'tau':best_t,'validation_mcc':best_m,'n_valid':len(valid),'n_invalid':len(rows)-len(valid),
            'score_coverage':len(valid)/len(rows),'degenerate':best_m is None,'grid_results':grid}

def inference_identity(manifest):
    return {'model_sha256':manifest['model_sha256'],'tokenizer':manifest['tokenizer'],
            'tokenizer_revision':manifest['tokenizer_revision'],'quantization':manifest['quantization'],
            'backend':'llama-cpp-python','backend_version':manifest['llama_cpp_python'],
            'scoring':manifest['scoring'],'seed_recorded':42,'context_tokens':manifest['context_tokens']}

def validate_validation(expected,records,arm,dataset,model_sha256):
    exp={str(r['id']):r for r in expected}
    if len(exp)!=len(expected):raise ValueError('Duplicate expected validation IDs')
    seen={}
    for r in records:
        uid=str(r['id'])
        if r.get('split')!='validation' or r.get('dataset')!=dataset or r.get('arm')!=arm:
            raise ValueError('Only the requested arm/dataset validation records may be fitted')
        if uid in seen:raise ValueError('Duplicate validation records')
        e=exp.get(uid)
        if e is None or r.get('label')!=e['label'] or r.get('prompt_sha256')!=e['prompt_sha256']:
            raise ValueError('Validation ID/label/prompt identity mismatch')
        if r.get('model_sha256')!=model_sha256 or r.get('seed')!=42:
            raise ValueError('Validation model/seed mismatch')
        seen[uid]=r
    return exp,seen

def make_lock(expected,records,arm,dataset,run_manifest,validation_prompt_set_sha256,source_evidence=None):
    exp,seen=validate_validation(expected,records,arm,dataset,run_manifest['model_sha256'])
    if set(exp)!=set(seen):raise ValueError(f'Incomplete validation: {len(seen)}/{len(exp)}; do not fit on partial data')
    ordered=[seen[uid] for uid in sorted(seen)]
    result=pick_threshold(ordered)
    evidence=[{k:r.get(k) for k in ('id','label','prompt_sha256','model_sha256','seed','s','valid')} for r in ordered]
    # Normalize invalid scores for strict JSON; their raw validity remains recorded.
    for r in evidence:
        if r['s'] is not None and not math.isfinite(r['s']):r['s']=None
    return {'arm':arm,'dataset':dataset,'split_used':'validation','locked_at_utc':datetime.now(timezone.utc).isoformat(),
            'rules_sha256':RULES_SHA256,'identity':inference_identity(run_manifest),
            'validation_prompt_set_sha256':validation_prompt_set_sha256,'n_expected':len(exp),
            'validation_evidence_sha256':digest_json(evidence),'source_evidence':source_evidence or [],
            'temporal_claim':'locked at actual time; does not certify that external test inference had not started',**result}

def persist_lock(path,entry):
    path=Path(path)
    data=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'schema':'thresholds_v1','rules':RULES,'rules_sha256':RULES_SHA256,'entries':{}}
    if data.get('rules_sha256')!=RULES_SHA256:raise ValueError('Threshold rules differ; create a new version')
    key=f"{entry['arm']}/{entry['dataset']}";old=data['entries'].get(key)
    if old:
        for k in ('rules_sha256','identity','validation_prompt_set_sha256','validation_evidence_sha256','tau'):
            if old.get(k)!=entry.get(k):raise ValueError(f'Locked threshold is immutable; conflicting {k} for {key}')
        return old
    data['entries'][key]=entry;write_json(path,data);return entry

def require_test_lock(path,arm,dataset,run_manifest,validation_prompt_set_sha256=None):
    path=Path(path)
    if not path.exists():raise ValueError('Test blocked: thresholds.json missing; complete validation first')
    data=json.loads(path.read_text(encoding='utf-8'));entry=data.get('entries',{}).get(f'{arm}/{dataset}')
    if not entry:raise ValueError(f'Test blocked: no locked threshold for {arm}/{dataset}')
    if data.get('rules_sha256')!=RULES_SHA256 or entry.get('rules_sha256')!=RULES_SHA256:
        raise ValueError('Test blocked: different threshold selection rules')
    if entry.get('identity')!=inference_identity(run_manifest):raise ValueError('Test blocked: model/tokenizer/backend differs from validation')
    if validation_prompt_set_sha256 and entry.get('validation_prompt_set_sha256')!=validation_prompt_set_sha256:
        raise ValueError('Test blocked: validation prompt set changed')
    return entry

def evaluate_at_locked_threshold(records,entry):
    valid=[r for r in records if r.get('valid') and isinstance(r.get('s'),(int,float)) and math.isfinite(r['s'])]
    y=[r['label'] for r in valid];pred=[int(r['s']>=entry['tau']) for r in valid]
    tp=sum(a==1 and b==1 for a,b in zip(y,pred));tn=sum(a==0 and b==0 for a,b in zip(y,pred))
    fp=sum(a==0 and b==1 for a,b in zip(y,pred));fn=sum(a==1 and b==0 for a,b in zip(y,pred))
    return {'tau':entry['tau'],'n':len(valid),'score_coverage':len(valid)/len(records) if records else 0,
            'mcc':mcc(y,pred),'accuracy':(tp+tn)/len(y) if y else None,
            'confusion_matrix':[[tn,fp],[fn,tp]],'threshold_locked_at_utc':entry['locked_at_utc'],
            'threshold_validation_evidence_sha256':entry['validation_evidence_sha256']}


In [ ]:
%%writefile /kaggle/working/kaggle_flip_frozen_runtime.py
"""Resume B2-flip on frozen donor prompts, repairing changed requests first.

The migration and scheduler run on CPU and are independently testable; only
run_batch loads the GPU model. Original checkpoint bytes remain in history/.
"""
import argparse
import hashlib
import json
import math
import os
import platform
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

try:
    import validation_lock as locks
except ModuleNotFoundError:
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent / 'Runs-management'))
    import validation_lock as locks

VARIANT = 'b2_flip_frozen_donor_v1'
STABLE = ('model_sha256', 'model_repo', 'quantization', 'llama_cpp_python', 'tokenizer',
          'tokenizer_revision', 'gpu_layers', 'context_tokens', 'scoring', 'limit')


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(b)
    return h.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def write_json_atomic(path, obj):
    locks.write_json(path, obj)


def read_rows(path):
    if not Path(path).exists():
        return []
    with Path(path).open(encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def write_rows_atomic(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8', newline='\n') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def load_bundle(path, snapshot=None):
    with zipfile.ZipFile(path) as z:
        manifest = json.loads(z.read('bundle_manifest.json'))
        payload = z.read('prompts.jsonl')
    if hashlib.sha256(payload).hexdigest() != manifest['prompts_sha256']:
        raise ValueError('Bundle prompt hash mismatch')
    if manifest.get('arm') != 'B2-flip' or manifest.get('prompt_variant') != VARIANT:
        raise ValueError('Use B2-flip bundles generated from the frozen donor reference')
    if snapshot and manifest['donor_snapshot_sha256'] != snapshot:
        raise ValueError('Wrong frozen donor snapshot')
    rows = [json.loads(line) for line in payload.splitlines() if line.strip()]
    if len(rows) != manifest['count'] or len({r['id'] for r in rows}) != len(rows):
        raise ValueError('Bundle count/unique ID mismatch')
    for row in rows:
        if (row['arm'], row['dataset'], row['split'], row['seed'], row['donor_snapshot_sha256']) != (
                'B2-flip', manifest['dataset'], manifest['split'], 42, manifest['donor_snapshot_sha256']):
            raise ValueError('Prompt identity mismatch')
        if hashlib.sha256(row['prompt'].encode()).hexdigest() != row['prompt_sha256']:
            raise ValueError('Prompt text/hash mismatch')
        if row['tokens']['total'] + 64 > manifest['prompt_budget_tokens']:
            raise ValueError('Prompt exceeds context budget')
    return manifest, rows


def validate_record(row, expected, identity, allow_legacy=False):
    if (row.get('arm'), row.get('dataset'), row.get('split'), row.get('label'), row.get('seed'), row.get('model_sha256')) != (
            'B2-flip', expected['dataset'], expected['split'], expected['label'], 42, identity['model_sha256']):
        raise ValueError(f'Checkpoint record identity mismatch: {row.get("id")}')
    is_current = row.get('prompt_sha256') == expected['prompt_sha256']
    is_legacy = row.get('prompt_sha256') == expected['previous_prompt_sha256']
    if not is_current and not (allow_legacy and is_legacy):
        raise ValueError(f'Unknown checkpoint prompt hash: {row["id"]}')
    demo_ids = expected['demo_ids'] if is_current else expected['previous_demo_ids']
    if row.get('demo_ids') != demo_ids:
        raise ValueError(f'Checkpoint demo provenance mismatch: {row["id"]}')
    return is_current


def prepare_session(runs, root, checkpoint, identity, snapshot):
    """Import only compatible records; return the immutable repair queue plus current state."""
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    contexts = {}
    for dataset, split, bundle, output in runs:
        m, rr = load_bundle(bundle, snapshot)
        if (m['dataset'], m['split']) != (dataset, split):
            raise ValueError('Run/bundle identity mismatch')
        if identity['model_sha256'] != m['model_sha256'] or identity['llama_cpp_python'] != m['expected_llama_cpp_python']:
            raise ValueError('Model bytes/backend version differ from reusable scores')
        for runtime_key, manifest_key in (('tokenizer', 'tokenizer'), ('tokenizer_revision', 'tokenizer_revision'),
                                          ('context_tokens', 'prompt_budget_tokens'), ('quantization', 'quantization'),
                                          ('model_repo', 'model_repo')):
            if identity[runtime_key] != m[manifest_key]:
                raise ValueError('Bundle tokenizer/context/model identity differs between splits')
        contexts[dataset, split] = {'manifest': m, 'rows': rr, 'expected': {r['id']: r for r in rr},
            'out': Path(output), 'identity': dict(identity, dataset=dataset, split=split,
                prompt_set_sha256=m['prompts_sha256'], bundle_sha256=sha256(bundle),
                donor_snapshot_sha256=snapshot, prompt_variant=VARIANT)}
    prompt_sets = {f'{d}/{s}': c['manifest']['prompts_sha256'] for (d, s), c in contexts.items()}
    signature = {'snapshot': snapshot, 'prompt_sets': prompt_sets, 'identity': {k: identity[k] for k in STABLE}}
    ready = root / 'migration_ready.json'
    if ready.exists():
        if read_json(ready) != signature:
            raise ValueError('Existing working output belongs to different prompts/runtime; use a new OUT_ROOT')
        plan = read_json(root / 'migration_plan.json')
    else:
        intent = root / 'migration_in_progress.json'
        source_sha = sha256(checkpoint) if checkpoint else None
        intended = dict(signature, source_checkpoint_sha256=source_sha)
        if intent.exists():
            if read_json(intent) != intended:
                raise ValueError('Interrupted migration used different inputs')
        elif any(root.rglob('records.jsonl')):
            raise ValueError('Unrecognized existing output; use a separate OUT_ROOT')
        else:
            write_json_atomic(intent, intended)
        retained = {key: [] for key in contexts}
        stale = []
        repair = []
        source_completed = 0
        thresholds = None
        original_archive = None
        if checkpoint:
            with zipfile.ZipFile(checkpoint) as z:
                if z.testzip() is not None or len(z.namelist()) != len(set(z.namelist())):
                    raise ValueError('Invalid or duplicate checkpoint ZIP members')
                native = 'migration_plan.json' in z.namelist()
                if native:
                    plan = json.loads(z.read('migration_plan.json'))
                    if plan['signature'] != signature:
                        raise ValueError('Native checkpoint belongs to a different frozen prompt/runtime version')
                    repair = plan['repair_requests']
                    if 'thresholds.json' in z.namelist():
                        thresholds = json.loads(z.read('thresholds.json'))
                    if 'history/original_checkpoint.zip' in z.namelist():
                        original_archive = z.read('history/original_checkpoint.zip')
                    if 'history/stale_records.jsonl' in z.namelist():
                        stale = [json.loads(line) for line in z.read('history/stale_records.jsonl').splitlines() if line.strip()]
                else:
                    original_archive = Path(checkpoint).read_bytes()
                allowed_records = {f'{d}/{s}/records.jsonl' for d, s in contexts}
                if any(n.endswith('/records.jsonl') and n not in allowed_records for n in z.namelist()):
                    raise ValueError('Checkpoint includes unexpected records paths')
                for key, c in contexts.items():
                    d, s = key
                    member = f'{d}/{s}/records.jsonl'
                    if member not in z.namelist():
                        continue
                    previous = json.loads(z.read(f'{d}/{s}/run_manifest.json'))
                    if any(previous.get(k) != identity[k] for k in STABLE):
                        raise ValueError(f'Checkpoint runtime differs for {d}/{s}')
                    expected_set = c['manifest']['prompts_sha256'] if native else c['manifest']['previous_prompts_sha256']
                    if previous.get('prompt_set_sha256') != expected_set:
                        raise ValueError(f'Unexpected checkpoint prompt set: {d}/{s}')
                    seen = set()
                    for line in z.read(member).splitlines():
                        if not line.strip():
                            continue
                        row = json.loads(line)
                        uid = row['id']
                        if uid in seen or uid not in c['expected']:
                            raise ValueError('Duplicate/unexpected checkpoint ID')
                        seen.add(uid)
                        source_completed += 1
                        item = c['expected'][uid]
                        current = validate_record(row, item, identity, allow_legacy=not native)
                        if current:
                            retained[key].append(row)
                        else:
                            stale.append(row)
                            repair.append({'dataset': d, 'split': s, 'id': uid,
                                           'old_prompt_sha256': row['prompt_sha256'], 'new_prompt_sha256': item['prompt_sha256']})
        if not checkpoint or not native:
            plan = {'schema': 'b2_flip_frozen_migration_v1', 'signature': signature,
                    'source_checkpoint_sha256': source_sha, 'source_completed': source_completed,
                    'initial_reused': sum(len(rr) for rr in retained.values()),
                    'repair_requests': repair, 'created_utc': datetime.now(timezone.utc).isoformat()}
        for request in repair:
            item = contexts[request['dataset'], request['split']]['expected'][request['id']]
            if request['new_prompt_sha256'] != item['prompt_sha256']:
                raise ValueError('Repair queue/new prompt mismatch')
        if len({(r['dataset'], r['split'], r['id']) for r in repair}) != len(repair):
            raise ValueError('Duplicate repair requests')
        history = root / 'history'
        history.mkdir(exist_ok=True)
        if original_archive:
            (history / 'original_checkpoint.zip').write_bytes(original_archive)
        write_rows_atomic(history / 'stale_records.jsonl', stale)
        write_json_atomic(root / 'migration_plan.json', plan)
        for key, c in contexts.items():
            c['out'].mkdir(parents=True, exist_ok=True)
            write_rows_atomic(c['out'] / 'records.jsonl', retained[key])
            write_json_atomic(c['out'] / 'run_manifest.json', c['identity'])
        if thresholds is not None:
            write_json_atomic(root / 'thresholds.json', thresholds)
        write_json_atomic(ready, signature)
    if plan.get('schema') != 'b2_flip_frozen_migration_v1' or plan.get('signature') != signature:
        raise ValueError('Migration plan version/signature mismatch')
    for request in plan['repair_requests']:
        item = contexts[request['dataset'], request['split']]['expected'][request['id']]
        if request['new_prompt_sha256'] != item['prompt_sha256']:
            raise ValueError('Persisted repair request differs from current prompt')
    for key, c in contexts.items():
        rr = read_rows(c['out'] / 'records.jsonl')
        c['done'] = {}
        for row in rr:
            uid = row['id']
            if uid in c['done'] or uid not in c['expected']:
                raise ValueError('Duplicate/unexpected working checkpoint ID')
            validate_record(row, c['expected'][uid], identity)
            c['done'][uid] = row
        previous = read_json(c['out'] / 'run_manifest.json')
        if any(previous.get(k) != c['identity'][k] for k in STABLE + ('prompt_set_sha256', 'donor_snapshot_sha256')):
            raise ValueError('Working run manifest changed')
    return {'root': root, 'contexts': contexts, 'plan': plan}


def process_session(session, infer, stop_at, clock=time.time):
    root, contexts, plan = session['root'], session['contexts'], session['plan']
    status_path = root / 'checkpoint_status.json'
    thresholds = root / 'thresholds.json'
    repairs = plan['repair_requests']
    locked = {}
    status = {'state': 'running', 'timed_out': False, 'phase': 'repair_changed_prompts',
              'started_at': datetime.now(timezone.utc).isoformat(), 'stop_new_records_at_epoch': stop_at,
              'source_completed': plan['source_completed'], 'initial_reused': plan['initial_reused'],
              'repair_total': len(repairs), 'inferred_this_session': 0, 'runs': {}}

    def persist():
        status['repair_remaining'] = sum(r['id'] not in contexts[r['dataset'], r['split']]['done'] for r in repairs)
        status['repair_completed'] = len(repairs) - status['repair_remaining']
        for (d, s), c in contexts.items():
            status['runs'][f'{d}/{s}'] = {'count': len(c['rows']), 'completed': len(c['done']),
                'state': 'complete' if len(c['done']) == len(c['rows']) else 'checkpointed', 'output': str(c['out'])}
        status['completed_records_total'] = sum(len(c['done']) for c in contexts.values())
        write_json_atomic(status_path, status)

    def lock_completed_validation(dataset):
        if dataset in locked:
            return
        c = contexts[dataset, 'validation']
        if len(c['done']) != len(c['rows']):
            return
        entry = locks.make_lock(c['rows'], list(c['done'].values()), 'B2-flip', dataset, c['identity'],
            c['manifest']['prompts_sha256'], [{'path': str(c['out'] / 'records.jsonl'),
            'records_sha256': sha256(c['out'] / 'records.jsonl')}])
        locked[dataset] = locks.persist_lock(thresholds, entry)
        write_json_atomic(c['out'] / 'threshold_lock.json', locked[dataset])
        print(f'Validation locked: {dataset} tau={entry["tau"]}; full validation n={entry["n_expected"]}', flush=True)

    def run_one(d, s, uid, phase):
        c = contexts[d, s]
        if uid in c['done']:
            return True
        if clock() >= stop_at:
            status.update(state='timed_out', timed_out=True, stopped_after_run=f'{d}/{s}',
                          timed_out_at=datetime.now(timezone.utc).isoformat())
            persist()
            return False
        if s == 'test':
            lock_completed_validation(d)
            locks.require_test_lock(thresholds, 'B2-flip', d, c['identity'],
                                    contexts[d, 'validation']['manifest']['prompts_sha256'])
        status['phase'] = phase
        persist()
        item = c['expected'][uid]
        row = infer(item, c)
        validate_record(row, item, c['identity'])
        row.update(donor_snapshot_sha256=c['manifest']['donor_snapshot_sha256'], prompt_variant=VARIANT,
                   inference_phase=phase)
        path = c['out'] / 'records.jsonl'
        with path.open('a', encoding='utf-8', newline='\n') as f:
            f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
            f.flush()
            os.fsync(f.fileno())
        c['done'][uid] = row
        status['inferred_this_session'] += 1
        status['last_checkpoint_at'] = datetime.now(timezone.utc).isoformat()
        if s == 'validation':
            lock_completed_validation(d)
        persist()
        print(f'{d}/{s} {uid}: s={row["s"]:.4f}, raw={row["raw"]!r} ({len(c["done"])}/{len(c["rows"])}); phase={phase}', flush=True)
        return True

    persist()
    try:
        for dataset, split in contexts:
            if split == 'validation':
                lock_completed_validation(dataset)
        # Repair all validation requests, then satisfy validation dependencies for repaired test requests.
        for r in repairs:
            if r['split'] == 'validation' and not run_one(r['dataset'], r['split'], r['id'], 'repair_changed_prompts'):
                return status
        for (d, s), c in contexts.items():
            if s == 'validation':
                for row in c['rows']:
                    if not run_one(d, s, row['id'], 'complete_validation_before_test'):
                        return status
        for r in repairs:
            if r['split'] == 'test' and not run_one(r['dataset'], r['split'], r['id'], 'repair_changed_prompts'):
                return status
        for (d, s), c in contexts.items():
            for row in c['rows']:
                if not run_one(d, s, row['id'], 'continue_missing_prompts'):
                    return status
        for (d, s), c in contexts.items():
            rr = list(c['done'].values())
            metrics = evaluate_output(c['out'] / 'records.jsonl')
            metrics['score_threshold_validation_locked'] = locks.evaluate_at_locked_threshold(rr, locked[d])
            write_json_atomic(c['out'] / 'evaluation.json', metrics)
        status.update(state='complete', phase='complete', completed_at=datetime.now(timezone.utc).isoformat())
        persist()
        return status
    except Exception as exc:
        status.update(state='error', error=f'{type(exc).__name__}: {exc}')
        persist()
        raise


def binary_scores(logits, label_ids):
    import numpy as np
    values = np.asarray(logits, dtype=np.float64).reshape(-1)
    if not np.all(np.isfinite(values)):
        raise ValueError('Non-finite logits')
    maximum = np.max(values)
    normalizer = float(maximum + np.log(np.exp(values - maximum).sum()))
    lp0, lp1 = (float(values[i] - normalizer) for i in label_ids)
    score = float(np.exp(lp1 - np.logaddexp(lp0, lp1)))
    best = int(np.argmax(values))
    top = np.argsort(values)[-5:][::-1]
    return {'logprob_0': lp0, 'logprob_1': lp1, 'p0': math.exp(lp0), 'p1': math.exp(lp1),
            's': score, 'pred_argmax': int(score >= .5), 'greedy_token_id': best,
            'top5': [{'token_id': int(i), 'logprob': float(values[i]-normalizer)} for i in top]}


def evaluate_output(path):
    from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
        confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score)
    rr = read_rows(path)
    actual = [r for r in rr if r.get('response_valid') and r.get('raw') in ('0', '1')]
    scores = [r for r in rr if r.get('valid') and isinstance(r.get('s'), (int, float)) and math.isfinite(r['s'])]
    metrics = {'n_total': len(rr), 'n_score_valid': len(scores), 'n_response_valid': len(actual),
        'response_coverage': len(actual)/len(rr) if rr else None, 'accuracy': None,
        'balanced_accuracy': None, 'precision': None, 'recall': None, 'f1': None, 'mcc': None,
        'confusion_matrix': None, 'auroc': None, 'auprc': None}
    if actual:
        y, p = [r['label'] for r in actual], [int(r['raw']) for r in actual]
        metrics.update(accuracy=accuracy_score(y, p), balanced_accuracy=balanced_accuracy_score(y, p),
            precision=precision_score(y, p, zero_division=0), recall=recall_score(y, p, zero_division=0),
            f1=f1_score(y, p, zero_division=0), mcc=matthews_corrcoef(y, p),
            confusion_matrix=confusion_matrix(y, p, labels=[0, 1]).tolist())
    if len({r['label'] for r in scores}) == 2:
        metrics.update(auroc=roc_auc_score([r['label'] for r in scores], [r['s'] for r in scores]),
            auprc=average_precision_score([r['label'] for r in scores], [r['s'] for r in scores]))
    return metrics


def run_batch(runs, model_path, model_revision, gpu_layers, deadline_epoch,
              checkpoint_reserve_seconds, status_path, checkpoint, snapshot):
    from llama_cpp import Llama, __version__ as llama_version
    from transformers import AutoTokenizer
    model_path = Path(model_path)
    model_digest = sha256(model_path)
    first, _ = load_bundle(runs[0][2], snapshot)
    identity = {'model_sha256': model_digest, 'model_repo': first['model_repo'],
                'quantization': first['quantization'], 'model_file': model_path.name,
                'llama_cpp_python': llama_version, 'model_repo_revision': f'local-sha256:{model_digest}',
                'tokenizer': first['tokenizer'], 'tokenizer_revision': first['tokenizer_revision'],
                'gpu_layers': gpu_layers, 'context_tokens': first['prompt_budget_tokens'],
                'scoring': 'unconstrained_first_token_logits_v1', 'limit': None, 'python': platform.python_version()}
    session = prepare_session(runs, Path(status_path).parent, checkpoint, identity, snapshot)
    stop_at = deadline_epoch - checkpoint_reserve_seconds
    print(f'Migration: initial compatible={session["plan"]["initial_reused"]}; repair queue={len(session["plan"]["repair_requests"])}', flush=True)
    # No new GPU work is started after the deadline.
    if time.time() >= stop_at or all(len(c['done']) == len(c['rows']) for c in session['contexts'].values()):
        return process_session(session, lambda *_: None, stop_at)
    tokenizer = AutoTokenizer.from_pretrained(first['tokenizer'], revision=first['tokenizer_revision'])
    llm = Llama(model_path=str(model_path), n_ctx=first['prompt_budget_tokens'], n_batch=256,
                n_gpu_layers=gpu_layers, logits_all=True, seed=42, verbose=True)
    label_ids = []
    for label in ('0', '1'):
        ids = tokenizer.encode(label, add_special_tokens=False)
        if len(ids) != 1 or ids != llm.tokenize(label.encode(), add_bos=False, special=False):
            raise ValueError('HF/GGUF label token mismatch')
        label_ids.append(ids[0])

    def infer(item, context):
        prompt = item['prompt']
        messages = [{'role': 'user', 'content': prompt}]
        chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        hf_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
        gguf_ids = llm.tokenize(chat.encode(), add_bos=False, special=True)
        if list(hf_ids) != gguf_ids or len(gguf_ids) + 64 > context['manifest']['prompt_budget_tokens']:
            raise ValueError('HF/GGUF chat tokenizer mismatch or context overflow')
        start = time.monotonic()
        llm.reset()
        llm.eval(gguf_ids)
        scores = binary_scores(llm.scores[llm.n_tokens-1], label_ids)
        raw = llm.detokenize([scores['greedy_token_id']]).decode('utf-8', errors='replace').strip()
        row = {k: item[k] for k in ('id', 'dataset', 'split', 'label', 'prompt_sha256', 'demo_ids', 'tokens', 'graph_missing')}
        row.update(scores, arm='B2-flip', raw=raw, response_valid=raw in ('0', '1'), valid=True,
                   token_ids={'0': label_ids[0], '1': label_ids[1]}, model_sha256=model_digest,
                   backend='llama-cpp-python', quantization=first['quantization'], model_repo=first['model_repo'],
                   model_repo_revision=identity['model_repo_revision'], seed=42,
                   timestamp=datetime.now(timezone.utc).isoformat(), logprob_distribution='unconstrained_first_token',
                   elapsed_s=time.monotonic()-start)
        return row

    return process_session(session, infer, stop_at)


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--model', required=True)
    ap.add_argument('--model-revision', default='local-input')
    ap.add_argument('--gpu-layers', type=int, default=-1)
    ap.add_argument('--deadline-epoch', type=float, required=True)
    ap.add_argument('--checkpoint-reserve-seconds', type=int, default=300)
    ap.add_argument('--status', required=True)
    ap.add_argument('--checkpoint')
    ap.add_argument('--snapshot-sha256', required=True)
    ap.add_argument('--run', action='append', nargs=4, required=True)
    args = ap.parse_args()
    run_batch(args.run, args.model, args.model_revision, args.gpu_layers, args.deadline_epoch,
              args.checkpoint_reserve_seconds, args.status, args.checkpoint, args.snapshot_sha256)


In [ ]:
# Runtime tự chuyển checkpoint; không extract checkpoint cũ đè lên output mới.
OUT_ROOT.mkdir(parents=True, exist_ok=True)
STATUS_PATH = OUT_ROOT/'checkpoint_status.json'
RUN_DEADLINE_EPOCH = time.time() + MAX_RUNTIME_HOURS*3600
cmd = [sys.executable, '-u', '/kaggle/working/kaggle_flip_frozen_runtime.py',
       '--model', model_path, '--gpu-layers', str(GPU_LAYERS),
       '--deadline-epoch', str(RUN_DEADLINE_EPOCH), '--checkpoint-reserve-seconds', str(CHECKPOINT_RESERVE_SECONDS),
       '--status', str(STATUS_PATH), '--snapshot-sha256', EXPECTED_DONOR_SHA]
if RESUME_FROM_CHECKPOINT:
    cmd += ['--checkpoint', str(CHECKPOINT_INPUT_PATH)]
for dataset, split in EXPECTED_RUNS:
    cmd += ['--run', dataset, split, str(BUNDLE_PATHS[dataset,split]), str(OUT_ROOT/dataset/split)]
try:
    run_logged(cmd, 'inference_batch')
finally:
    # Cả timeout/lỗi backend đều giữ những record đã fsync để có thể tiếp tục.
    if (OUT_ROOT/'migration_ready.json').is_file():
        checkpoint_archive = shutil.make_archive('/kaggle/working/grace_b2_flip_checkpoint', 'zip', root_dir=OUT_ROOT)
        print('Checkpoint phiên donor mới:', checkpoint_archive)
    else:
        print('Chưa chuyển xong checkpoint; sửa lỗi đầu vào rồi dùng lại checkpoint gốc đã gắn.')
    diagnostics_zip()
if STATUS_PATH.exists():
    print(STATUS_PATH.read_text(encoding='utf-8'))


In [ ]:
status = json.loads((OUT_ROOT/'checkpoint_status.json').read_text(encoding='utf-8'))
print('Trạng thái:', status['state'])
print('Đã có output tương thích:', status['completed_records_total'], '/ 10011')
print('Đã sửa prompt:', status['repair_completed'], '/', status['repair_total'])
print('LLM calls phiên này:', status['inferred_this_session'])
print('Tải grace_b2_flip_checkpoint.zip để tiếp tục phiên sau nếu chưa hoàn tất.')
diagnostics_zip()


Ngưỡng nằm trong `OUT_ROOT/thresholds.json` của **phiên donor mới**. Runtime khóa từ validation đầy đủ và kiểm tra trước mỗi test; không sao chép ngưỡng phiên cũ. Model/tokenizer/backend/seed/hash checkpoint sai sẽ dừng. Bản checkpoint I-06 gốc được giữ trong `history/original_checkpoint.zip`, record cũ bị thay trong `history/stale_records.jsonl`.
